# **Data Preparation Notebook **

## Objectives

* Prepare data

## Inputs


## Outputs




---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/workspaces/heritage-housing-issues-milestone/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/workspaces/heritage-housing-issues-milestone'

# Data Preparation



In [4]:
import pandas as pd
df = pd.read_csv(f"outputs/datasets/collection/house_prices")
df.head()

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,GarageArea,GarageFinish,GarageYrBlt,...,LotArea,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,548,RFn,2003.0,...,8450,65.0,196.0,61,5,7,856,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,460,RFn,1976.0,...,9600,80.0,0.0,0,8,6,1262,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,608,RFn,2001.0,...,11250,68.0,162.0,42,5,7,920,2001,2002,223500
3,961,NaN,NaN,No,216,ALQ,540,642,Unf,1998.0,...,9550,60.0,0.0,35,5,7,756,1915,1970,140000
4,1145,NaN,4.0,Av,655,GLQ,490,836,RFn,2000.0,...,14260,84.0,350.0,84,5,8,1145,2000,2000,250000


---

# Missing Values

In [5]:
# Check sum of null values for each variable
df.isnull().sum()

1stFlrSF          0
2ndFlrSF         86
BedroomAbvGr     99
BsmtExposure     38
BsmtFinSF1        0
BsmtFinType1    145
BsmtUnfSF         0
GarageArea        0
GarageFinish    235
GarageYrBlt      81
GrLivArea         0
KitchenQual       0
LotArea           0
LotFrontage     259
MasVnrArea        8
OpenPorchSF       0
OverallCond       0
OverallQual       0
TotalBsmtSF       0
YearBuilt         0
YearRemodAdd      0
SalePrice         0
dtype: int64

In [7]:
# Display a table to visualise null data
null_data = []

for col in df.columns:
    null_count = df[col].isnull().sum()
    if null_count > 0:
        null_data.append({
            'Variable': col,
            'Null Count': null_count,
            'Null %': round(null_count / len(df) * 100, 1),
            'Data Type': 'Numerical' if df[col].dtype in ['int64', 'float64'] else 'Categorical'
        })

null_df = pd.DataFrame(null_data).sort_values('Null %', ascending=False)
print(null_df.to_string(index=False))

    Variable  Null Count  Null %   Data Type
 LotFrontage         259    17.7   Numerical
GarageFinish         235    16.1 Categorical
BsmtFinType1         145     9.9 Categorical
BedroomAbvGr          99     6.8   Numerical
    2ndFlrSF          86     5.9   Numerical
 GarageYrBlt          81     5.5   Numerical
BsmtExposure          38     2.6 Categorical
  MasVnrArea           8     0.5   Numerical


We will be imputing missing data for the following reasons:
* To avoid loss of statistical power, compared to dropping rows with null values
* To reduce bias
* Ensure compatability with algorithms

We have chosen to impute median values for numerical values. The reasoning behind this is housing attributes data is likely to be skewed, with a small number of large properties affecting the mode. For this reason, we will be using median.

With categorical values, we will be imputing the mode value.



In [8]:
from sklearn.impute import SimpleImputer

# Median imputation for numerical variables
median_imputer = SimpleImputer(strategy='median')
median_cols = ['LotFrontage', 'GarageYrBlt', 'BedroomAbvGr', '2ndFlrSF', 'MasVnrArea']
df[median_cols] = median_imputer.fit_transform(df[median_cols])

# Mode imputation for categorical variables
mode_imputer = SimpleImputer(strategy='most_frequent')
mode_cols = ['BsmtExposure', 'BsmtFinType1', 'GarageFinish']
df[mode_cols] = mode_imputer.fit_transform(df[mode_cols])

# Confirm no null values remain
print(df.isnull().sum())

1stFlrSF        0
2ndFlrSF        0
BedroomAbvGr    0
BsmtExposure    0
BsmtFinSF1      0
BsmtFinType1    0
BsmtUnfSF       0
GarageArea      0
GarageFinish    0
GarageYrBlt     0
GrLivArea       0
KitchenQual     0
LotArea         0
LotFrontage     0
MasVnrArea      0
OpenPorchSF     0
OverallCond     0
OverallQual     0
TotalBsmtSF     0
YearBuilt       0
YearRemodAdd    0
SalePrice       0
dtype: int64


---

---

# Push files to Repo